# RAG: Implementation (Production)

In [10]:
from openai import OpenAI
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import CrossEncoder

In [11]:
from google.colab import userdata
api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

## Load PDF

In [12]:
def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

document = load_pdf("/content/Agreement_for_sale.pdf")

## Chunking

In [13]:
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []

    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap

    return chunks

chunks = chunk_text(document)
print("Total chunks:", len(chunks))

Total chunks: 66


## Embeddings

In [14]:
def embed(texts):
    return [
        client.embeddings.create(
            model="text-embedding-3-small",
            input=t
        ).data[0].embedding
        for t in texts
    ]

embeddings = embed(chunks)

## FAISS Vector DB

In [15]:
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension) # By default, FAISS uses Euclidean distance (L2)

index.add(np.array(embeddings).astype("float32"))

## Retrieval

In [16]:
def retrieve(query, k=10):
    q_emb = embed([query])[0]
    D, I = index.search(np.array([q_emb]).astype("float32"), k)

    return [(chunks[i], D[0][j]) for j, i in enumerate(I[0])]

## Re-ranking (Cross-Encoder)

In [17]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, retrieved):
    chunk_texts = [c for c, _ in retrieved]
    pairs = [[query, c] for c in chunk_texts]

    scores = reranker.predict(pairs)

    reranked = list(zip(chunk_texts, scores))
    return sorted(reranked, key=lambda x: x[1], reverse=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## Build Prompt

In [18]:
def answer(query):
    retrieved = retrieve(query, k=10)
    reranked = rerank(query, retrieved)[:3]

    context = "\n".join([c for c, _ in reranked])

    prompt = f"""
Answer using ONLY the context below.

Context:
{context}

Question:
{query}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    return response.output_text

## RAG in Action

In [19]:
query = "What's the maintenance charges of the society?"

print(answer(query))

The provisional maintenance charge is Rs. 5/- per square foot of carpet area plus adjoining balcony area per month, charged for 24 months in advance.


In [20]:
#!pip install faiss-cpu

In [21]:
#!pip install pypdf